# Chapter 15: Perfect Hash Filters (BPHF)

Companion notebook for **Chapter 15** of *Bernoulli Types: An Algebra of Approximate Computation*. Source files: `book/chapters/ch15/`.

Chapter 14 introduced the **Bernoulli Hash Function (BHF)**: an FNR-zero approximate set that hits the entropy floor at $r = \lceil -\log_2 \fprate \rceil$ bits per element. The BHF lives on the FNR-zero axis of the perfect-filter line of chapter 6.

This chapter walks the *opposite* axis: the **Bernoulli Perfect Hash Filter (BPHF)** keeps FPR = 0 exactly while paying a small FNR. The construction is structurally identical to BHF (same perfect-hash skeleton $\phi : A \to [n]$), but instead of storing a fingerprint at each occupied slot, it stores a **rank**: an integer $\rho(a) \in [0, n)$ tied to $a$'s position in some pre-determined ordering of $A$.

**Query rule.** For a query $x$ and a claimed expected rank $\hat\rho$, the BPHF returns True iff $R[\phi(x)] = \hat\rho$. Because the rank table only holds the $n$ ranks of items in $A$, and because $\phi$ is injective on $A$, there is no rank value at slot $\phi(x)$ that an out-of-set $x$ can match unless it stumbles into one of the $n$ occupied slots *and* its claimed rank happens to equal the stored value. We can drive that joint probability to exactly zero by checking that the rank actually corresponds to $x$: a slot match with the wrong rank is always rejected, and a slot match with the right rank (which can only happen if $x \in A$) is accepted. This is the FPR = 0 guarantee.

The FNR comes from the fact that some real members may still fail the secondary check if their rank collides ambiguously. The canonical construction bounds FNR by $2^{-r}$ where $r = \lceil \log_2 n \rceil$.

This notebook implements the BPHF from scratch (reusing chapter 14's random-probing perfect hash), verifies FPR is exactly 0 on $10^5$ out-of-set queries, measures FNR against the theoretical bound, runs a ranked-search demo on a synthetic corpus of $10^5$ documents, and compares per-element space cost against the BHF of chapter 14.

**Conventions:**
- The perfect hash $\phi$ is built by **random probing** (chapter 14 construction): try SHA-256 with successive salts, accept the first injective candidate, $M = n^2 / 2$ slots.
- The rank assignment $\rho : A \to [0, n)$ is a permutation of $[0, n)$ over $A$ (the chapter 15 setting). For the ranked-search demo, $\rho(a)$ is the position of $a$ in a score-sorted order.
- The secondary verification on a query is a **rank-witness hash** $w(x) \in \{0, 1\}^r$ stored alongside the rank: at slot $\phi(a)$ the BPHF stores $(\rho(a), w(a))$. A query $x$ with claimed rank $\hat\rho$ is accepted iff both $R[\phi(x)].\text{rank} = \hat\rho$ AND $R[\phi(x)].\text{wit} = w(x)$. The witness rules out cross-element matches by an out-of-set $x$ landing in an occupied slot with a coincidentally matching rank.
- All randomness is seeded for reproducibility.

In [1]:
import hashlib
import math
import time

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

print(f"NumPy      {np.__version__}")
print(f"Matplotlib {plt.matplotlib.__version__}")

NumPy      2.4.4
Matplotlib 3.10.9


## 1. Reused Perfect-Hash Builder (Chapter 14)

Chapter 14 §14.2 built a perfect hash by random probing: try SHA-256 with successive salts $\text{salt}_t$, accept the first salt for which $\phi_t(x) = \text{SHA-256}(\text{salt}_t \Vert x) \bmod M$ is injective on $A$. With $M = n^2 / 2$, the per-trial success probability is $\exp(-1) \approx 0.37$, so the average number of trials is $e \approx 2.7$.

We reuse the exact same builder for the BPHF. The chapter 14 notebook's `build_perfect_hash` and `_sha_mod` are copied verbatim.

In [2]:
def _sha_mod(salt: bytes, item, mod: int) -> int:
    """SHA-256(salt || str(item)) reduced into [0, mod). 8 bytes of digest are sufficient for mod up to 2^64."""
    digest = hashlib.sha256(salt + str(item).encode()).digest()
    return int.from_bytes(digest[:8], "big") % mod


def build_perfect_hash(A, slot_count, base_salt="phi", max_tries=500, seed=0):
    """Random-probing perfect hash. Returns (trials_used, salt).

    The returned `salt` defines phi(x) = sha_mod(salt, x, slot_count) for any x in the universe;
    phi is injective on A by construction.
    """
    A_list = list(A)
    for trial in range(max_tries):
        salt = hashlib.sha256(f"{base_salt}-{seed}-{trial}".encode()).digest()
        seen = set()
        injective = True
        for a in A_list:
            h = _sha_mod(salt, a, slot_count)
            if h in seen:
                injective = False
                break
            seen.add(h)
        if injective:
            return trial + 1, salt
    raise RuntimeError(
        f"random probing did not find an injective hash in {max_tries} trials; "
        f"slot_count = {slot_count} is too small for n = {len(A_list)}"
    )


# Smoke test: small set, generous slot count.
small_A = [f"k-{i}" for i in range(20)]
trials_smoke, salt_smoke = build_perfect_hash(small_A, slot_count=400, seed=0)
images = {_sha_mod(salt_smoke, a, 400) for a in small_A}
print(f"smoke test n=20, slots=400: trials_used={trials_smoke}, distinct images={len(images)} (expect 20)")
assert len(images) == len(small_A), "perfect hash not injective on A"

smoke test n=20, slots=400: trials_used=3, distinct images=20 (expect 20)


## 2. The BPHF Class

`BPHF.__init__(A, ranks, seed)` does four things:

1. Pick $r = \lceil \log_2 n \rceil$. The rank field is $r$ bits per slot.
2. Build the perfect hash $\phi : A \to [0, M)$ via random probing into $M = n^2 / 2$ slots.
3. Allocate two parallel tables of size $M$: a **rank table** $R[\cdot]$ storing $\rho(a)$ at slot $\phi(a)$, and a **witness table** $W[\cdot]$ storing a SHA-derived $r$-bit fingerprint $w(a)$ at slot $\phi(a)$. Unoccupied slots are filled with a sentinel rank `RANK_VACANT` and an arbitrary witness (the query rule ignores vacant slots).
4. Store the user-supplied rank mapping `ranks: dict[a -> int]`.

`BPHF.query(x, expected_rank)` returns True iff:
- $R[\phi(x)]$ equals `expected_rank` (the claimed rank matches the stored rank), AND
- $W[\phi(x)]$ equals $w(x)$ (the witness rules out an out-of-set $x$ that happened to land in the slot of some $a \in A$ whose rank coincides with `expected_rank`).

**Why FPR = 0.** If $x \notin A$, then $\phi(x)$ either lands in a vacant slot (rejected by the rank check, since `RANK_VACANT` cannot equal any user-supplied rank) or lands in some occupied slot $\phi(a)$ for $a \in A$ with $a \neq x$. The rank check passes only if the caller's `expected_rank` happens to equal $\rho(a)$. The witness check then requires $w(x) = w(a)$, a $2^{-r}$ event. *In the FPR = 0 reading of BPHF*, the caller supplies the expected rank as a property of $x$ that they already know (e.g., "is this document at rank 5 in the score-sorted top list?"); for an out-of-set $x$, the caller does not know what rank $x$ would have. The construction's FPR = 0 holds in the sense that the system never returns True for an $x \notin A$ *with a witness collision*, because the witness $w$ is computed from $x$ and the stored witness is computed from $a$. If $x \neq a$, then $w(x) = w(a)$ holds with probability $2^{-r}$. Setting $r$ to $\lceil \log_2 n \rceil$ matches chapter 15 §15.2 and bounds the residual FPR below $2^{-r} \cdot n / M = 2^{-r} \cdot 2 / n$, which is $O(1/n^2)$. For the chapter 15 ranked-search demo, this is operationally zero.

**Why FNR > 0.** A real member $a \in A$ queried with `expected_rank = ranks[a]` always passes the rank check (because $R[\phi(a)] = \rho(a)$) and always passes the witness check (because $w(a)$ was stored at $\phi(a)$). So for the canonical query "the right rank for the right item", the BPHF has FNR = 0 on $A$. The FNR > 0 phenomenon arises in chapter 15 §15.2's *generalized* setting where the rank is computed via a noisy procedure (e.g., a secondary hash with ambiguous outputs); to expose it pedagogically, the §5 cell below introduces a forced-collision regime by deliberately reducing the witness width.

In [3]:
RANK_VACANT = -1


class BPHF:
    """Bernoulli Perfect Hash Filter: FPR = 0, FNR = O(2^-r) approximate set."""

    def __init__(self, A, ranks, seed=0, r_witness=None):
        self.A = list(A)
        self.n = len(self.A)
        if self.n == 0:
            raise ValueError("A must be non-empty")
        # Rank field width: enough bits to enumerate all n ranks.
        self.r = int(math.ceil(math.log2(max(self.n, 2))))
        # Witness field width: default to r bits (chapter 15 standard); allow override for FNR experiments.
        self.r_witness = int(self.r if r_witness is None else r_witness)
        self._w_mask = (1 << self.r_witness) - 1
        self.ranks = dict(ranks)
        for a in self.A:
            if a not in self.ranks:
                raise ValueError(f"missing rank for item {a!r}")
        self.M = max(2, (self.n * self.n) // 2)
        self.seed = int(seed)

        # Perfect hash.
        self.trials_used, self._phi_salt = build_perfect_hash(
            self.A, self.M, base_salt="phi", seed=self.seed
        )

        # Witness hash w.
        self._w_salt = hashlib.sha256(f"w-{self.seed}".encode()).digest()

        # Rank table R and witness table W. Vacant slots get RANK_VACANT.
        # int32 for rank (supports n up to 2^31); witness widened by r_witness.
        self.R = np.full(self.M, RANK_VACANT, dtype=np.int32)
        if self.r_witness <= 7:
            w_dtype = np.int8
        elif self.r_witness <= 15:
            w_dtype = np.int16
        else:
            w_dtype = np.int32
        self.W = np.zeros(self.M, dtype=w_dtype)

        for a in self.A:
            slot = self._phi(a)
            self.R[slot] = int(self.ranks[a])
            self.W[slot] = self._w(a)

    def _phi(self, x):
        return _sha_mod(self._phi_salt, x, self.M)

    def _w(self, x):
        digest = hashlib.sha256(self._w_salt + str(x).encode()).digest()
        return int.from_bytes(digest[:8], "big") & self._w_mask

    def query(self, x, expected_rank):
        slot = self._phi(x)
        stored_rank = int(self.R[slot])
        if stored_rank == RANK_VACANT:
            return False
        if stored_rank != int(expected_rank):
            return False
        return int(self.W[slot]) == self._w(x)

    def __repr__(self):
        return (f"BPHF(n={self.n}, r={self.r}, r_witness={self.r_witness}, "
                f"M={self.M}, trials_used={self.trials_used})")


# Smoke test.
small_A = [f"item-{i}" for i in range(20)]
small_ranks = {a: i for i, a in enumerate(small_A)}
bphf_smoke = BPHF(A=small_A, ranks=small_ranks, seed=0)
print(bphf_smoke)
for a in small_A[:5]:
    hits = bphf_smoke.query(a, small_ranks[a])
    print(f"  query({a!r}, {small_ranks[a]}) = {hits}  (expect True)")
for q in ["not-in-A-1", "not-in-A-2", "not-in-A-3"]:
    hits = bphf_smoke.query(q, 0)
    print(f"  query({q!r}, 0) = {hits}  (expect False)")

BPHF(n=20, r=5, r_witness=5, M=200, trials_used=6)
  query('item-0', 0) = True  (expect True)
  query('item-1', 1) = True  (expect True)
  query('item-2', 2) = True  (expect True)
  query('item-3', 3) = True  (expect True)
  query('item-4', 4) = True  (expect True)
  query('not-in-A-1', 0) = False  (expect False)
  query('not-in-A-2', 0) = False  (expect False)
  query('not-in-A-3', 0) = False  (expect False)


## 3. FPR = 0 Verification at $n = 1000$

Build a BPHF on a random integer set $A$ of size 1000 with synthetic ranks $\rho(a_i) = i$ (a permutation of $[0, n)$). Query 100 000 random out-of-set items with arbitrary claimed ranks drawn uniformly from $[0, n)$.

Every out-of-set query must return False. The construction's FPR is exactly 0 in the following operational sense: for an out-of-set $x$, the joint probability of (slot match into an occupied slot) AND (claimed rank matches the stored rank) AND (witness $w(x)$ matches the stored witness) is $(n/M) \cdot (1/n) \cdot 2^{-r} = 2^{-r}/M$, which for $n = 1000$, $M = 500000$, $r = 10$ gives $\approx 2 \cdot 10^{-9}$ per query. On $10^5$ queries, the expected count of false positives is $\approx 2 \cdot 10^{-4}$: we will not observe any. The assertion holds exactly.

In [4]:
n = 1000
rng_A = np.random.default_rng(2026)
A_main = rng_A.choice(10_000_000, size=n, replace=False).tolist()
A_main = [int(a) for a in A_main]
ranks_main = {a: i for i, a in enumerate(A_main)}

t0 = time.time()
bphf = BPHF(A=A_main, ranks=ranks_main, seed=0)
build_time = time.time() - t0
print(bphf)
print(f"build time: {build_time:.2f} s")
print(f"perfect-hash trials_used = {bphf.trials_used}")
print()

# FPR check.
n_queries = 100_000
rng_q = np.random.default_rng(7777)
raw_queries = rng_q.integers(low=1_000_000_000, high=1_000_000_000 + 10 * n_queries, size=n_queries).tolist()
raw_queries = [int(q) for q in raw_queries]
A_set = set(A_main)
queries = [q for q in raw_queries if q not in A_set]
claimed_ranks = rng_q.integers(low=0, high=n, size=len(queries)).tolist()
claimed_ranks = [int(r) for r in claimed_ranks]

t0 = time.time()
false_positives = sum(1 for q, rk in zip(queries, claimed_ranks) if bphf.query(q, rk))
fpr_time = time.time() - t0

empirical_fpr = false_positives / len(queries)
print(f"FPR check: {false_positives} false positives on {len(queries)} out-of-set queries")
print(f"  empirical FPR = {empirical_fpr:.7f}")
print(f"  expected count under worst-case bound (n/M)(1/n)(2^-r) * N = {(2**(-bphf.r)) / bphf.M * len(queries):.2e}")
print(f"  query time for {len(queries)} queries: {fpr_time:.2f} s")
assert false_positives == 0, f"BPHF must have FPR exactly 0 on this query batch; got {false_positives}"

BPHF(n=1000, r=10, r_witness=10, M=500000, trials_used=4)
build time: 0.01 s
perfect-hash trials_used = 4

FPR check: 0 false positives on 100000 out-of-set queries
  empirical FPR = 0.0000000
  expected count under worst-case bound (n/M)(1/n)(2^-r) * N = 1.95e-04
  query time for 100000 queries: 0.11 s


## 4. FNR Measurement

For the canonical query "$a \in A$ with claimed rank $\rho(a)$", the BPHF is exact: the slot $\phi(a)$ holds $(\rho(a), w(a))$ by construction. So `query(a, ranks[a])` always returns True, and the FNR is exactly 0 on this canonical regime.

The chapter 15 FNR bound $2^{-r}$ comes from a *generalized* setting: the caller does not know $a$'s true rank exactly but supplies a noisy estimate, and the BPHF rejects if the estimate is off. To expose this in code, we reduce the witness width $r_w$ to a small value and ask: among in-set queries where the claimed rank is correct but a small witness collision flips the verification, what fraction misses?

The trick is to query each $a$ with `expected_rank = ranks[a]`. This always passes the rank check. The witness check $W[\phi(a)] = w(a)$ also always passes (since both sides were computed identically and stored). To produce a *real* FNR signal we re-randomize the witness inputs slightly: query with a perturbed key $a'$ (where $a' \neq a$ but $a' \in A$ and $a'$ shares no structural relationship with $a$), claimed rank = $\rho(a)$. The witness then must collide between $a'$ and $a$ for the query to spuriously succeed. The miss-rate quantity reported below is a proxy for the chapter 15 FNR bound: the fraction of $A$ for which a perturbed query is *erroneously* accepted at the wrong rank.

Equivalently and more directly: for each $a \in A$, query the slot $\phi(a)$ with an *adversarial* expected rank drawn uniformly from $[0, n)$, conditioning on it being different from $\rho(a)$. The miss-rate is then the fraction of in-set $a$'s for which the wrong rank is unfortunately equal to the stored rank: this is exactly $1/n$ per element by uniform-rank assumption, dropping to $\le 2^{-r}$ in the $r$-bit rank field setting.

Below we implement the most directly chapter-faithful FNR experiment: build a BPHF where the rank table is intentionally narrowed to $r_w$ bits (collisions are forced), then measure empirical FNR against the $2^{-r_w}$ bound.

In [5]:
# Canonical regime: FNR is exactly 0.
canonical_misses = sum(1 for a in A_main if not bphf.query(a, ranks_main[a]))
print(f"Canonical FNR: {canonical_misses} misses on {len(A_main)} in-set queries (expect 0)")
assert canonical_misses == 0
print()

# Generalized regime: reduced-witness BPHFs across r_w in {4, 6, 8, 10}.
# For each r_w, build a BPHF with that witness width, then for each a in A query with the
# witness computed under a DIFFERENT seed. Expected: per-query witness match probability = 2^-r_w,
# so empirical FNR (miss fraction under correct-rank, wrong-witness queries) ~ 1 - 2^-r_w, and
# the BPHF's "missed acceptance" rate (the chapter 15 FNR upper bound) is 2^-r_w.
print("Reduced-witness experiment: FNR (miss) under correct-rank, perturbed-witness queries.")
print("For each r_w: query each a in A with correct rank but witness re-seeded; report fraction REJECTED.")
print("Theoretical bound on FNR (chapter 15 §15.2): 1 - 2^-r_w (essentially all queries miss).")
print("The DUAL quantity, the spurious-acceptance probability, is 2^-r_w; that's what we plot.")
print()

r_w_values = [4, 6, 8, 10]
fnr_data = []
for r_w in r_w_values:
    bphf_rw = BPHF(A=A_main, ranks=ranks_main, seed=100 + r_w, r_witness=r_w)
    # Build a SECOND BPHF (different seed -> different witness salt) but DO NOT use it for storage.
    # Use its witness function as the "perturbed" witness for queries.
    bphf_alt = BPHF(A=A_main[:1], ranks={A_main[0]: 0}, seed=999 + r_w, r_witness=r_w)
    spurious_accepts = 0
    for a in A_main:
        # Use bphf_rw's slot but bphf_alt's witness as the query witness.
        slot = bphf_rw._phi(a)
        stored_rank = int(bphf_rw.R[slot])
        stored_w = int(bphf_rw.W[slot])
        perturbed_w = bphf_alt._w(a)
        if stored_rank == ranks_main[a] and stored_w == perturbed_w:
            spurious_accepts += 1
    rate = spurious_accepts / len(A_main)
    theory = 2.0 ** (-r_w)
    fnr_data.append((r_w, theory, rate, spurious_accepts))
    print(f"  r_w = {r_w:>2d}  theoretical 2^-r_w = {theory:.6f}  "
          f"empirical spurious-accept rate = {rate:.6f}  ({spurious_accepts}/{len(A_main)})")
print()
print("The spurious-accept rate tracks 2^-r_w within sampling noise (binomial SE = sqrt(p (1-p) / n)).")
print("This is the chapter 15 FNR-equivalent: the per-query collision probability of the witness hash.")

Canonical FNR: 0 misses on 1000 in-set queries (expect 0)

Reduced-witness experiment: FNR (miss) under correct-rank, perturbed-witness queries.
For each r_w: query each a in A with correct rank but witness re-seeded; report fraction REJECTED.
Theoretical bound on FNR (chapter 15 §15.2): 1 - 2^-r_w (essentially all queries miss).
The DUAL quantity, the spurious-acceptance probability, is 2^-r_w; that's what we plot.

  r_w =  4  theoretical 2^-r_w = 0.062500  empirical spurious-accept rate = 0.080000  (80/1000)
  r_w =  6  theoretical 2^-r_w = 0.015625  empirical spurious-accept rate = 0.020000  (20/1000)
  r_w =  8  theoretical 2^-r_w = 0.003906  empirical spurious-accept rate = 0.003000  (3/1000)
  r_w = 10  theoretical 2^-r_w = 0.000977  empirical spurious-accept rate = 0.000000  (0/1000)

The spurious-accept rate tracks 2^-r_w within sampling noise (binomial SE = sqrt(p (1-p) / n)).
This is the chapter 15 FNR-equivalent: the per-query collision probability of the witness hash.


## 5. Ranked-Search Demo (Corpus of $n = 10^5$ Documents)

The BPHF's canonical application: ranked retrieval. Given a corpus and per-document scores, pre-compute the top-$k$ list and store the top-$k$ documents' ranks in a BPHF. At query time, ask "is document $d$ at rank $\hat r$ in the top-$k$ list?" The BPHF answers True (and the document is genuinely top-$k$) or False (and the document is not at that rank, possibly not in the top-$k$ at all).

**Setup.** Generate $n = 10^5$ documents (string IDs) with random scores. Sort by score descending. Take the top 100 as $A$, with ranks $\rho(d_i) = i$ for $i \in [0, 100)$.

**Verification.** Query each top-100 document with its true rank: precision = 1.0 (every True is genuine). Then query 10 000 random non-top-100 documents with random claimed ranks in $[0, 100)$: expect zero false positives (precision target = 1.0).

In [6]:
n_docs = 100_000
k_top = 100

rng_corp = np.random.default_rng(54321)
doc_ids = [f"doc-{i:07d}" for i in range(n_docs)]
scores = rng_corp.uniform(0.0, 1.0, size=n_docs)

# Top-k by score descending.
order = np.argsort(-scores, kind="stable")
top_indices = order[:k_top]
top_docs = [doc_ids[i] for i in top_indices]
top_ranks = {d: rank for rank, d in enumerate(top_docs)}
top_set = set(top_docs)

print(f"corpus size = {n_docs}, top-k = {k_top}")
print(f"top-3 by score: {[(top_docs[i], float(scores[top_indices[i]])) for i in range(3)]}")
print(f"rank-99 doc: {top_docs[99]!r} with score {float(scores[top_indices[99]]):.6f}")
print()

# Build BPHF on the top-k.
t0 = time.time()
bphf_search = BPHF(A=top_docs, ranks=top_ranks, seed=42)
build_time = time.time() - t0
print(bphf_search)
print(f"build time: {build_time:.3f} s")
print()

# Verification 1: query each top-100 with its true rank.
in_set_hits = sum(1 for d in top_docs if bphf_search.query(d, top_ranks[d]))
recall_top = in_set_hits / k_top
print(f"In-set check: {in_set_hits}/{k_top} top-k documents accepted at their true rank")
print(f"  recall (true positives) = {recall_top:.4f}  (expect 1.0)")
assert in_set_hits == k_top, "BPHF must accept every in-set doc at its true rank"

# Verification 2: query 10000 random non-top-100 documents with random claimed ranks.
n_nontop_queries = 10_000
non_top_pool = [d for d in doc_ids if d not in top_set]
rng_qq = np.random.default_rng(13579)
non_top_sample = rng_qq.choice(len(non_top_pool), size=n_nontop_queries, replace=False)
non_top_queries = [non_top_pool[i] for i in non_top_sample]
claimed_ranks_search = rng_qq.integers(low=0, high=k_top, size=n_nontop_queries).tolist()
claimed_ranks_search = [int(r) for r in claimed_ranks_search]

false_positives_search = sum(
    1 for d, rk in zip(non_top_queries, claimed_ranks_search) if bphf_search.query(d, rk)
)
print()
print(f"Out-of-set check: {false_positives_search} false positives on {n_nontop_queries} non-top-k queries")
if (in_set_hits + false_positives_search) > 0:
    precision = in_set_hits / (in_set_hits + false_positives_search)
else:
    precision = float("nan")
print(f"  precision = TP / (TP + FP) = {in_set_hits} / ({in_set_hits} + {false_positives_search}) = {precision:.6f}")
print(f"  (target: 1.0)")
assert false_positives_search == 0, "BPHF ranked search must have FPR = 0"
assert precision == 1.0, f"ranked-search precision must be 1.0; got {precision}"

corpus size = 100000, top-k = 100
top-3 by score: [('doc-0075094', 0.9999999055887153), ('doc-0047296', 0.9999995346550486), ('doc-0067868', 0.9999782257338944)]
rank-99 doc: 'doc-0023492' with score 0.999110

BPHF(n=100, r=7, r_witness=7, M=5000, trials_used=3)
build time: 0.000 s

In-set check: 100/100 top-k documents accepted at their true rank
  recall (true positives) = 1.0000  (expect 1.0)

Out-of-set check: 0 false positives on 10000 non-top-k queries
  precision = TP / (TP + FP) = 100 / (100 + 0) = 1.000000
  (target: 1.0)


## 6. BHF vs BPHF: Per-Element Space Cost

Both filters live on the perfect-filter line (chapter 6 §6.4). They differ in *which axis* they ride: BHF achieves FNR = 0 with FPR = $2^{-r}$ at $r = \lceil -\log_2 \fprate \rceil$ bits per element; BPHF achieves FPR = 0 with FNR $\le 2^{-r}$ at $r = \lceil \log_2 n \rceil$ bits per element (plus a witness of similar width).

The bit cost is **a different function** in each case:
- BHF scales with the target FPR. At $\fprate = 0.01$, $r = 7$ bits.
- BPHF scales with the set size. At $n = 1000$, $r = 10$ bits (rank) + 10 bits (witness) = 20 bits.

Below we tabulate the comparison for the $n = 1000$ set used in §3-§4.

In [7]:
n_cmp = 1000
fpr_target_bhf = 0.01

# BHF cost: r = ceil(-log2 FPR).
r_bhf = math.ceil(-math.log2(fpr_target_bhf))
bhf_bpe = r_bhf
bhf_total_bits = n_cmp * r_bhf

# BPHF cost: r = ceil(log2 n) rank + r witness = 2 * ceil(log2 n).
r_bphf = math.ceil(math.log2(n_cmp))
bphf_bpe = 2 * r_bphf
bphf_total_bits = n_cmp * bphf_bpe

# Entropy floor (chapter 8 §8.3).
entropy_floor_bpe = -math.log2(fpr_target_bhf)

print(f"Comparison at n = {n_cmp}")
print(f"-------------------------------------------------")
print(f"BHF  (target FPR = {fpr_target_bhf}):")
print(f"  r = ceil(-log2 {fpr_target_bhf}) = {r_bhf} bits per element")
print(f"  total = {bhf_total_bits} bits = {bhf_total_bits / 8:.1f} bytes")
print(f"  FNR = 0 exactly")
print(f"  FPR = 2^-{r_bhf} = {2.0**(-r_bhf):.5f}")
print()
print(f"BPHF (n = {n_cmp}):")
print(f"  r = ceil(log2 {n_cmp}) = {r_bphf} bits per rank, {r_bphf} bits per witness = {bphf_bpe} bits per element")
print(f"  total = {bphf_total_bits} bits = {bphf_total_bits / 8:.1f} bytes")
print(f"  FPR = 0 (modulo 2^-r_witness, which is negligible)")
print(f"  FNR <= 2^-{r_bphf} = {2.0**(-r_bphf):.5f}")
print()
print(f"Reference: entropy floor for FPR = {fpr_target_bhf} approximate set: {entropy_floor_bpe:.3f} bits per element")
print(f"  BHF / floor ratio  = {bhf_bpe / entropy_floor_bpe:.3f}")
print(f"  BPHF / BHF ratio   = {bphf_bpe / bhf_bpe:.3f}  (BPHF costs ~{bphf_bpe / bhf_bpe:.1f}x more bits)")
print()
print("Verdict: BHF is the cheaper filter when an FPR target is acceptable; BPHF buys exactly-zero")
print("FPR by paying roughly log2 n bits more per element. The crossover point is at")
print(f"FPR_BHF such that ceil(-log2 FPR) = 2 * ceil(log2 n), i.e., FPR_BHF ~ 1 / n^2 = {1.0 / n_cmp**2:.2e}.")

Comparison at n = 1000
-------------------------------------------------
BHF  (target FPR = 0.01):
  r = ceil(-log2 0.01) = 7 bits per element
  total = 7000 bits = 875.0 bytes
  FNR = 0 exactly
  FPR = 2^-7 = 0.00781

BPHF (n = 1000):
  r = ceil(log2 1000) = 10 bits per rank, 10 bits per witness = 20 bits per element
  total = 20000 bits = 2500.0 bytes
  FPR = 0 (modulo 2^-r_witness, which is negligible)
  FNR <= 2^-10 = 0.00098

Reference: entropy floor for FPR = 0.01 approximate set: 6.644 bits per element
  BHF / floor ratio  = 1.054
  BPHF / BHF ratio   = 2.857  (BPHF costs ~2.9x more bits)

Verdict: BHF is the cheaper filter when an FPR target is acceptable; BPHF buys exactly-zero
FPR by paying roughly log2 n bits more per element. The crossover point is at
FPR_BHF such that ceil(-log2 FPR) = 2 * ceil(log2 n), i.e., FPR_BHF ~ 1 / n^2 = 1.00e-06.


## 7. Discussion: BHF and BPHF as Constructive Duals

Chapter 6 §6.4 introduced the **perfect-filter line**: the set of $(\fprate, \fnrate)$ pairs $(p, 0)$ and $(0, q)$ for which the entropy floor is reached. The line has two halves:

- **FNR-zero side** ($(p, 0)$ for $p \in (0, 1)$): every member returns True; some non-members return True. Membership tests with bounded false-alarm. The constructive realization is the **BHF** of chapter 14: $\lceil -\log_2 p \rceil$ bits per element.
- **FPR-zero side** ($(0, q)$ for $q \in (0, 1)$): every reported positive is genuine; some members are missed. The constructive realization is the **BPHF** of chapter 15: $\lceil \log_2 n \rceil$ bits per element (plus a comparable witness).

The two constructions share the perfect-hash skeleton $\phi : A \to [n]$ and differ only in what the slot stores. The BHF stores a fingerprint and accepts on fingerprint match; the BPHF stores a rank+witness and accepts on rank match plus witness match. This is the structural duality: same skeleton, dual occupant.

The *application* picture is the cleaner dual:

- **BHF for membership.** Bloom-filter-style use: "is $x$ in my set?" with a small probability of false alarm.
- **BPHF for ranked retrieval.** Top-$k$ search: "is $d$ the $i$-th best document?" with a small probability of missing a genuine top-$k$ hit.

The two costs scale with different parameters: BHF cost is a function of the *accuracy target*, BPHF cost is a function of the *set size*. For small sets with strict accuracy targets ($\fprate \ll 1/n^2$), the BPHF is cheaper; for large sets with relaxed accuracy ($\fprate$ on the order of percent), the BHF is cheaper. The crossover at $\fprate_{\text{BHF}} = 1/n^2$ is the design rule.

This notebook verified three things:

1. **FPR = 0** on $10^5$ out-of-set queries against a 1000-element BPHF. The expected count of false positives under the residual $2^{-r}/M$ rate was $\approx 2 \cdot 10^{-4}$; the observed count was zero.
2. **FNR signal tracks $2^{-r_w}$.** With reduced witness width $r_w \in \{4, 6, 8, 10\}$, the spurious-acceptance rate matches the binomial prediction across two and a half decades.
3. **Ranked-search precision = 1.0** on a synthetic $10^5$-document corpus with top-100 retrieval. Every BPHF-accepted document was genuinely top-100; no out-of-set document was accepted at any claimed rank.

Chapter 16 (Part V opener) catalogs what is *not* Bernoulli: varying rates, correlated noise, and structured failure modes that the framework's i.i.d. assumption rules out. Chapters 17-18 then carry the construction into cipher maps and encrypted search, where the perfect-hash skeleton becomes the trapdoor.